# Whisper STT: Meeting Sample (WAV)

Same pipeline as the Project Nimbus notebook, applied to `recordings/meeting_sample.wav`: an informal, multi-speaker meeting with laughter, cross-talk and background noise.

## Step 1: Setting Up the Environment

In [2]:
!uv pip install openai pydub python-dotenv

Using Python 3.12.14 environment at: /Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/.venv
Checked 3 packages in 30ms


In [3]:
import io
import os
import shutil
import time
import difflib
import json
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

AUDIO_DIR = Path("recordings")
CHUNKS_DIR = Path("chunks")          # chunks/<recording name>/chunk_000.mp3 ...
OUTPUT_DIR = Path("output")
for d in (AUDIO_DIR, CHUNKS_DIR, OUTPUT_DIR):
    d.mkdir(exist_ok=True)

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ffmpeg found:", shutil.which("ffmpeg"))
print("Audio files:", [f.name for f in AUDIO_DIR.glob("*")])

API key loaded: True
ffmpeg found: /opt/homebrew/bin/ffmpeg
Audio files: ['project_nimbus_meeting.mp3', 'meeting_sample.wav']


## Step 2: Sample Meeting Audio

In [4]:
from pydub import AudioSegment
from IPython.display import Audio, display

AUDIO_PATH = AUDIO_DIR / "meeting_sample.wav"
RECORDING_NAME = AUDIO_PATH.stem   # used for chunk folder and output file names
assert AUDIO_PATH.exists(), f"Audio file not found: {AUDIO_PATH}"

# pydub reads WAV directly via ffmpeg, no conversion needed
raw_audio = AudioSegment.from_file(AUDIO_PATH)
print(f"File:        {AUDIO_PATH.name}")
print(f"Size:        {AUDIO_PATH.stat().st_size / (1024 * 1024):.2f} MB (Whisper API limit: 25 MB)")
print(f"Duration:    {len(raw_audio) / 1000:.1f} s ({len(raw_audio) / 60000:.1f} min)")
print(f"Channels:    {raw_audio.channels}")
print(f"Sample rate: {raw_audio.frame_rate} Hz")

# Speech doesn't need stereo: downmix to mono (halves the data, same words)
meeting_audio = raw_audio.set_channels(1)
print(f"After downmix: {meeting_audio.channels} channel")

display(Audio(meeting_audio[:10_000].export(format="wav").read(), rate=meeting_audio.frame_rate))

File:        meeting_sample.wav
Size:        18.31 MB (Whisper API limit: 25 MB)
Duration:    300.0 s (5.0 min)
Channels:    2
Sample rate: 16000 Hz
After downmix: 1 channel


## Step 3: Basic Transcription (Without Chunking)

In [5]:
short_clip = meeting_audio[:60_000]


def to_buffer(segment, name="clip.mp3"):
    """Export a pydub AudioSegment to an in-memory mp3 that the API accepts."""
    buf = io.BytesIO()
    segment.export(buf, format="mp3")
    buf.seek(0)
    buf.name = name
    return buf


print("Transcribing first 60 seconds...")
basic_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=to_buffer(short_clip, "clip_60s.mp3"),
)

print("Response type:", type(basic_transcript).__name__)
print("Fields:", list(basic_transcript.model_dump().keys()))
print("\nTranscription:")
print("-" * 40)
print(basic_transcript.text)

Transcribing first 60 seconds...
Response type: Transcription
Fields: ['text', 'languages', 'logprobs', 'usage']

Transcription:
----------------------------------------
and you say, oh, we'll just do it up there. If you're talking about something simple, where you throw away a lot of these dimensions, then you can do that right away. We're talking about something that has all of these different facets that we have here. It won't happen quickly, it won't be easy, and there's all sorts of issues about keeping the equipment safe or else falling to the ground. Then maybe we should try to bring people here. I think our first priority should be to try to get people to come here. We're set up for it. The room is really underused. I thought the free lunch idea was a great idea. Yeah, I thought so too. Free lunch is good. I think we can get people to come here, but the issue is you definitely want to make sure that the kind of group you're getting is the right group so you don't waste a lot of

## Step 4: Transcription with Prompts (Guided Approach)

Whisper treats the prompt as preceding text. This prompt states the overall topic in one sentence, then lists the rare terms (Haas, P-files, Broadcast News) that Whisper might misspell. It deliberately avoids describing specific points of the conversation. Only the last ~224 tokens are used.

In [6]:
MEETING_PROMPT = (
    "Research group meeting about recruiting volunteer participants for recorded meetings "
    "and managing data storage and archiving. "
    # "Terms: non-linguists, Haas Business School, Broadcast News, P-files, CD-ROM."
)

print("Transcribing first 60 seconds WITH prompt...")
prompted_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=to_buffer(short_clip, "clip_60s.mp3"),
    prompt=MEETING_PROMPT,
)

print("\nTranscription (prompted):")
print("-" * 40)
print(prompted_transcript.text)

(OUTPUT_DIR / f"{RECORDING_NAME}_clip_prompted.txt").write_text(prompted_transcript.text)

Transcribing first 60 seconds WITH prompt...

Transcription (prompted):
----------------------------------------
If you're talking about something simple where you throw away a lot of these dimensions, then you can do that right away. If you're talking about something that has all of these different facets that we have here, it won't happen quickly, it won't be easy. There's all sorts of issues about keeping the equipment safe or else falling to the ground. So then maybe we should try to bring people here. I think our first priority should be to try to get people to come here. We're set up for it. The room is really underused. I thought the free lunch idea was a great idea. Yeah, I thought so too. Free lunch is good. I think we can get people to come here, but the issue is you definitely want to make sure that the kind of group that you're getting is the right group, so you don't waste a lot of your time in the overhead and bring people down. No crunchy food. Well, I was thinking lunch

1112

## Step 5: Transcription Without Prompts (Unguided Approach)

In [7]:
print("Transcribing first 60 seconds WITHOUT prompt...")
unprompted_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=to_buffer(short_clip, "clip_60s.mp3"),
)
(OUTPUT_DIR / f"{RECORDING_NAME}_clip_unprompted.txt").write_text(unprompted_transcript.text)

prompted_text = prompted_transcript.text
unprompted_text = unprompted_transcript.text

print("\nUNPROMPTED:\n" + unprompted_text)
print("\nPROMPTED:\n" + prompted_text)

a, b = unprompted_text.split(), prompted_text.split()
matcher = difflib.SequenceMatcher(None, a, b)

print("\nDifferences (unprompted -> prompted):")
print("-" * 40)
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag != "equal":
        print(f"{' '.join(a[i1:i2])!r:30} -> {' '.join(b[j1:j2])!r}")

print(f"\nSimilarity: {matcher.ratio():.1%}")

Transcribing first 60 seconds WITHOUT prompt...

UNPROMPTED:
and you say, oh, we'll just do it up there. If you're talking about something simple, where you throw away a lot of these dimensions, then you can do that right away. We're talking about something that has all of these different facets that we have here. It won't happen quickly, it won't be easy, and there's all sorts of issues about keeping the equipment safe or else falling to the ground. Then maybe we should try to bring people here. I think our first priority should be to try to get people to come here. We're set up for it. The room is really underused. I thought the free lunch idea was a great idea. Yeah, I thought so too. Free lunch is good. I think we can get people to come here, but the issue is you definitely want to make sure that the kind of group you're getting is the right group so you don't waste a lot of your time in the overhead and bring people down. No crunchy food. Well, I was thinking lunch afterwards. Lun

## Step 6: Implementing Audio Chunking

In [8]:
def split_audio(audio, out_dir, chunk_length_minutes=1.0, prefix="chunk"):
    """Split an AudioSegment into fixed-length mp3 chunks.

    Returns dicts with each chunk's path and start offset in seconds,
    which Step 7 needs to correct per-chunk timestamps.
    """
    chunk_ms = int(chunk_length_minutes * 60 * 1000)
    out_dir.mkdir(parents=True, exist_ok=True)

    for old in out_dir.glob(f"{prefix}_*.mp3"):
        old.unlink()

    chunks = []
    for i, start_ms in enumerate(range(0, len(audio), chunk_ms)):
        piece = audio[start_ms:start_ms + chunk_ms]
        path = out_dir / f"{prefix}_{i:03d}.mp3"
        piece.export(path, format="mp3")
        chunks.append({
            "index": i,
            "path": path,
            "offset": start_ms / 1000,
            "duration": len(piece) / 1000,
        })
    return chunks


chunk_dir = CHUNKS_DIR / RECORDING_NAME   # one subfolder per recording
chunk_info = split_audio(meeting_audio, chunk_dir, chunk_length_minutes=1)

print(f"Created {len(chunk_info)} chunks in '{chunk_dir}/'\n")
for c in chunk_info:
    size_kb = c["path"].stat().st_size / 1024
    print(f"{c['path'].name}  offset={c['offset']:6.1f}s  duration={c['duration']:5.1f}s  size={size_kb:6.1f} KB")

total = sum(c["duration"] for c in chunk_info)
print(f"\nTotal chunk duration: {total:.1f}s vs original {len(meeting_audio) / 1000:.1f}s")

Created 5 chunks in 'chunks_sample/'

chunk_000.mp3  offset=   0.0s  duration= 60.0s  size= 176.2 KB
chunk_001.mp3  offset=  60.0s  duration= 60.0s  size= 176.2 KB
chunk_002.mp3  offset= 120.0s  duration= 60.0s  size= 176.2 KB
chunk_003.mp3  offset= 180.0s  duration= 60.0s  size= 176.2 KB
chunk_004.mp3  offset= 240.0s  duration= 60.0s  size= 176.2 KB

Total chunk duration: 300.0s vs original 300.0s


## Step 7: Transcribing Chunks with Timestamps

In [9]:
def fmt_time(seconds):
    """Format seconds as HH:MM:SS."""
    s = int(seconds)
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}:{s % 60:02d}"


def transcribe_chunks(chunks, prompt=None, delay=0.5, retries=3):
    """Transcribe each chunk with timestamps and shift them to full-recording time."""
    segments, words = [], []

    for c in chunks:
        print(f"Transcribing {c['path'].name} ({c['index'] + 1}/{len(chunks)})...")

        for attempt in range(retries):
            try:
                with open(c["path"], "rb") as f:
                    result = client.audio.transcriptions.create(
                        model="whisper-1",
                        file=f,
                        response_format="verbose_json",
                        timestamp_granularities=["segment", "word"],
                        **({"prompt": prompt} if prompt else {}),
                    )
                break
            except Exception as e:
                if attempt == retries - 1:
                    raise
                wait = 2 ** (attempt + 1)
                print(f"  error ({e}); retrying in {wait}s")
                time.sleep(wait)

        # Timestamps are relative to the chunk, so add the chunk's offset
        for seg in result.segments:
            segments.append({
                "chunk": c["index"],
                "start": seg.start + c["offset"],
                "end": seg.end + c["offset"],
                "text": seg.text.strip(),
            })
        for w in result.words:
            words.append({
                "chunk": c["index"],
                "word": w.word,
                "start": w.start + c["offset"],
                "end": w.end + c["offset"],
            })

        time.sleep(delay)

    return segments, words


all_segments, all_words = transcribe_chunks(chunk_info, prompt=MEETING_PROMPT)

print(f"\n{len(all_segments)} segments, {len(all_words)} words\n")
for seg in all_segments:
    print(f"[{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['text']}")

starts = [s["start"] for s in all_segments]
print("\nMonotonic starts:", starts == sorted(starts))
print(f"Last segment ends at {all_segments[-1]['end']:.1f}s (recording is {len(meeting_audio) / 1000:.1f}s)")

Transcribing chunk_000.mp3 (1/5)...
Transcribing chunk_001.mp3 (2/5)...
Transcribing chunk_002.mp3 (3/5)...
Transcribing chunk_003.mp3 (4/5)...
Transcribing chunk_004.mp3 (5/5)...

91 segments, 907 words

[00:00:01 - 00:00:06] If you're talking about something simple where you throw away a lot of these dimensions, then you can do that right away.
[00:00:06 - 00:00:11] If you're talking about something that has all of these different facets that we have here, it won't happen quickly, it won't be easy.
[00:00:12 - 00:00:16] There's all sorts of issues about keeping the equipment safe or else falling to the ground.
[00:00:16 - 00:00:18] So then maybe we should try to bring people here.
[00:00:18 - 00:00:22] I think our first priority should be to try to get people to come here.
[00:00:22 - 00:00:27] We're set up for it. The room is really underused.
[00:00:28 - 00:00:30] I thought the free lunch idea was a great idea.
[00:00:30 - 00:00:31] Yeah, I thought so too.
[00:00:31 - 00:00:31] Fre

## Step 8: Exporting with Timestamps

In [10]:
def export_txt(segments, path):
    lines = [f"[{fmt_time(s['start'])} - {fmt_time(s['end'])}] {s['text']}" for s in segments]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def export_json(segments, words, path, source, prompt):
    data = {
        "source": str(source),
        "model": "whisper-1",
        "prompt": prompt,
        "duration": segments[-1]["end"] if segments else 0,
        "full_text": " ".join(s["text"] for s in segments),
        "segments": segments,
        "words": words,
    }
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


export_txt(all_segments, OUTPUT_DIR / f"{RECORDING_NAME}_transcript.txt")
export_json(all_segments, all_words, OUTPUT_DIR / f"{RECORDING_NAME}_transcript.json", AUDIO_PATH, MEETING_PROMPT)

# Checkpoint: files exist and are readable
for ext in ("txt", "json"):
    p = OUTPUT_DIR / f"{RECORDING_NAME}_transcript.{ext}"
    print(f"{p}  ({p.stat().st_size / 1024:.1f} KB)")

print("\nTXT preview:")
print("".join(open(OUTPUT_DIR / f"{RECORDING_NAME}_transcript.txt").readlines()[:5]))

output/sample_transcript.txt  (6.6 KB)
output/sample_transcript.srt  (7.6 KB)
output/sample_transcript.json  (123.7 KB)

SRT preview:
1
00:00:01,220 --> 00:00:06,080
If you're talking about something simple where you throw away a lot of these dimensions, then you can do that right away.

2
00:00:06,360 --> 00:00:11,800
If you're talking about something that has all of these different facets that we have here, it won't happen quickly, it won't be easy.

3
00:00:12,040 --> 00:00:16,520
There's all sorts of issues about keeping the equipment safe or else falling to the ground.


